1. Введіть своє прізвище та ім'я, а також номер варіанту

In [ ]:
student_name = "Прізвище ім'я"
variant_number = 1

2. Запустіть код, натисніть кнопку "Вибрати файли" та виберіть датасет, що відповідає номеру вашого варіанта

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
import os

def is_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False


if is_colab():
    print("📥 Запущено в Google Colab — оберіть файл для завантаження")

    from google.colab import files
    uploaded = files.upload()

    if uploaded:
        file_name = list(uploaded.keys())[0]
        print(f"Файл завантажено: {file_name}")
    else:
        print("Файл не було обрано")

# =========================
# 1. ЗАВАНТАЖЕННЯ ДАТАСЕТУ
# =========================

if os.path.exists("/content/" + file_name):
    file_path = "/content/" + file_name
else:
    file_path = file_name

df = pd.read_csv(file_path)

print("Початковий розмір:", df.shape)

target_column = "label"

y = df[target_column]
X = df.drop(columns=[target_column])

3. Поділ на тренувальний і тестовий датасет, обробка даних, графічна візуалізація результатів обробки та генерація .png-файлу

In [ ]:
# =========================
# 3. РОЗДІЛЕННЯ НА TRAIN / TEST ДАТАСЕТИ
# =========================

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# =========================
# 4. АНАЛІЗ ЧИСЛОВИХ І КАТЕГОРІАЛЬНИХ КОЛОНОК
# =========================

categorical_cols = X.select_dtypes(include=["object"]).columns
numeric_cols = X.select_dtypes(exclude=["object"]).columns

print("Категоріальні:", list(categorical_cols))
print("Числові:", list(numeric_cols))

# =========================
# 5. КОДУВАННЯ ТЕКСТОВИХ ОЗНАК
# =========================

encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

# навчання тільки на train
X_train_cat = encoder.fit_transform(X_train[categorical_cols])

# трансформація test через той самий encoder
X_test_cat = encoder.transform(X_test[categorical_cols])

# повертаємо в DataFrame
cat_feature_names = encoder.get_feature_names_out(categorical_cols)

X_train_cat = pd.DataFrame(X_train_cat, columns=cat_feature_names, index=X_train.index)
X_test_cat = pd.DataFrame(X_test_cat, columns=cat_feature_names, index=X_test.index)


# =========================
# 6. КОДУВАННЯ МІТКИ
# =========================

le = LabelEncoder()

y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

# =========================
# 7. МАСШТАБУВАННЯ ЧИСЛОВИХ ОЗНАК
# =========================

scaler = StandardScaler()

X_train_num = pd.DataFrame(
    scaler.fit_transform(X_train[numeric_cols]),
    columns=numeric_cols,
    index=X_train.index
)

X_test_num = pd.DataFrame(
    scaler.transform(X_test[numeric_cols]),
    columns=numeric_cols,
    index=X_test.index
)

# =========================
# 8. ОБ'ЄДНАННЯ
# =========================

X_train_processed = pd.concat([X_train_num, X_train_cat], axis=1)
X_test_processed = pd.concat([X_test_num, X_test_cat], axis=1)

train_final = X_train_processed.copy()
train_final["label"] = y_train

test_final = X_test_processed.copy()
test_final["label"] = y_test

# =========================
# 9. ЗБЕРЕЖЕННЯ CSV
# =========================

train_final.to_csv("train_processed.csv", index=False)
test_final.to_csv("test_processed.csv", index=False)

print("Готово!")
print("Train shape:", train_final.shape)
print("Test shape:", test_final.shape)

# =========================
# 10. ВІЗУАЛІЗАЦІЯ ТА СТВОРЕННЯ .PNG-ФАЙЛУ
# =========================

num_features = list(numeric_cols)[:3]
cat_features = list(categorical_cols)[:3]

total_rows = len(num_features) + len(cat_features)

fig, axs = plt.subplots(total_rows, 2, figsize=(14, 4 * total_rows))

fig.suptitle(
    f"ЛР2 — Попередня обробка даних\n"
     f"Датасет: {file_name}\n"
    f"Студент: {student_name} / Номер варіанту: {variant_number}",
    fontsize=16,
    y=0.98
)

plt.subplots_adjust(top=0.88, hspace=0.6, wspace=0.3)

row = 0

# =========================
# 10.1. АНАЛІЗ МАСШТАБУВАННЯ
# =========================

for feature in num_features:

    axs[row, 0].plot(X_train[feature].values[:100])
    axs[row, 0].set_title(f"{feature} — ДО масштабування")

    axs[row, 1].plot(X_train_num[feature].values[:100])
    axs[row, 1].set_title(f"{feature} — ПІСЛЯ масштабування")

    row += 1

# =========================
# 10.2. АНАЛІЗ КОДУВАННЯ
# =========================

encoding_info = []

for feature in cat_features:

    encoded = pd.get_dummies(X_train[feature])

    categories = encoded.columns.tolist()

    # ---- структура encoding ----
    axs[row, 0].barh(categories, [1] * len(categories))
    axs[row, 0].set_title(f"{feature} → {len(categories)} one-hot колонок")
    axs[row, 0].set_xlim(0, 1)

    # ---- частота значень ----
    freq = X_train[feature].value_counts()

    axs[row, 1].bar(freq.index.astype(str), freq.values)
    axs[row, 1].set_title(f"{feature} — частота значень")
    axs[row, 1].tick_params(axis='x', rotation=45)

    encoding_info.append({
        "feature": feature,
        "num_categories": len(categories),
        "categories": categories,
        "sparsity_%": round(100 * (1 - encoded.values.mean()), 2)
    })

    row += 1

# =========================
# 10.3. ЗБЕРЕЖЕННЯ PNG
# =========================

script_dir = os.getcwd()
output_path = os.path.join(script_dir, f"ЛР2 - {student_name} - Варіант {variant_number}.png")

plt.savefig(output_path, dpi=300, bbox_inches="tight")

plt.show()

# =========================
# 14. ТАБЛИЦЯ КОДУВАННЯ
# =========================

encoding_df = pd.DataFrame(encoding_info)

print("\n📊 STRUCTURE OF ONE-HOT ENCODING:\n")
print(encoding_df)

print(f"\n📁 Графік збережено у: {output_path}")